# 1. TIF 파일 자체(네이티브) 메타데이터 확인
해당 TIF 파일(칩이 아닌 전체 씬 원본) 내부에 박혀있는 기본 프로필 및 태그(Tags) 정보를 확인합니다.

In [1]:
import rasterio
import pprint
from pathlib import Path
import xml.etree.ElementTree as ET

# 칩(Chip)이 아닌 전체 씬(Scene) 원본 TIF 파일 경로
tif_path = Path('../data/interim/k3a_extracted/K3A_20150401044329_00095_00004060_L1R/K3A_20150401044329_00095_00004060_L1R_B.tif')

print(f"Reading TIF: {tif_path.name}\n")

with rasterio.open(tif_path) as src:
    print("=== 1. Basic Image Profile ===")
    pprint.pprint(src.profile)
    print("\n=== 2. All Namespaces Tags (Dataset Level) ===")
    # 모든 네임스페이스의 태그를 가져옵니다.
    for ns in src.tag_namespaces():
        print(f"\nNamespace: '{ns}'")
        pprint.pprint(src.tags(ns=ns))
    print("\n=== 3. Embedded RPCs/GCPs ===")
    print(f"RPCs inside TIF: {src.rpcs is not None}")
    print(f"GCPs inside TIF: {len(src.gcps[0]) > 0 if src.gcps else False}")


Reading TIF: K3A_20150401044329_00095_00004060_L1R_B.tif

=== 1. Basic Image Profile ===
{'blockxsize': 6015,
 'blockysize': 32,
 'count': 1,
 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 52N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",129],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]'),
 'driver': 'GTiff',
 'dtype': 'uint16',
 'height': 5820,
 'interleave': 'band',
 'nodata': None,
 'tiled': False,
 'transform': Affine(2.27940152, 0.0, 313602.84993858,
       0.0, -2.23081309, 4164685.39202369),
 'width': 6015}

=== 2. All Namespaces Tags (Dataset Level) ===

Namespace: 'IMAGE_STRUCTURE'
{'INTERLEAVE': 'BAND'}

Namespace: 

# 2. K3A 씬 전체(Scene) 메타데이터 (Aux.xml 및 RPC 파일)
TIF 파일 자체 메타데이터 외에, K3A가 제공하는 씬(Scene) 단위의 전체 메타데이터(Aux.xml) 및 RPC 파일을 확인합니다.

In [ ]:
scene_dir = tif_path.parent
aux_files = list(scene_dir.glob("*Aux.xml"))
rpc_files = list(scene_dir.glob("*_B_rpc.txt"))

print("=== 4. Scene Metadata (Aux.xml) ===")
if aux_files:
    tree = ET.parse(aux_files[0])
    root = tree.getroot()
    # 주요 씬 정보 출력 (General 태그 하위)
    general_info = root.find('.//General')
    if general_info is not None:
        for child in general_info:
            if len(child) == 0:  # 하위 태그가 없는 단일 텍스트
                print(f"{child.tag}: {child.text}")
else:
    print("Aux.xml not found.")
    
print("\n=== 5. RPC Text File ===")
if rpc_files:
    with open(rpc_files[0], 'r') as f:
        # 처음 10줄만 출력
        print(''.join(f.readlines()[:10]))
        print("...")
else:
    print("RPC txt file not found.")


=== 4. Scene Metadata (Aux.xml) ===
Satellite: KOMPSAT-3A
Sensor: AEISS-A
OrbitNumber: 95
OrbitDirection: ASCENDING
PassID: K3A_20150401044329_00095_00004060_L1R
ProductLevel: Level1R
ImageFormat: GEOTIFF
ImagingMode: Strip Imaging Mode
EllipsoidType: WGS-84
ResamplingMethod: TCLS
DesignBitsPerPixel: 14
ApplyMTFC: true
ApplyPODPAD: true
CreateDate: 20151202134803.00
ProductID: None
PMSVersionNo: CVT_LevelProcessor_v1.2

=== 5. RPC Text File ===
LINE_OFF:	+2909.88 pixels
SAMP_OFF:	+3007.50 pixels
LAT_OFF:	 +37.56431164 degrees
LONG_OFF:	+126.97660269 degrees
HEIGHT_OFF:	+2000.00 meters
LINE_SCALE:	+2909.88 pixels
SAMP_SCALE:	+3007.50 pixels
LAT_SCALE:	  +0.07130289 degrees
LONG_SCALE:	  +0.09682041 degrees
HEIGHT_SCALE:	+2000.00 meters

...
